# 19b · **eval** — `transfer` 150k ckpt × 5회 × 500 에피소드

`19a` 에서 학습한 것을 평가한다. **`TAGS`·`TASK` 를 19a 와 똑같이 맞출 것.**

- 평가 대상 = **150k 체크포인트 하나** (best-ckpt 안 고름 → 모델 간 공정).
- **rep 마다 `--seed = 1000 + 100·rep`** → env 초기상태가 달라진다.
  (같은 seed 로 5번 돌리면 결정적이라 반복이 무의미. 학습 seed=모델 분산 / rep=평가 분산)
- 끝난 run 은 자동 skip → 중단/재실행 안전.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

# ── 무엇을 / 어디서 ──────────────────────────────────────────────────────────
TASK = cf.SHORT_SIM         # ★ 'transfer' (AlohaTransferCube-v0).  insertion 은 cf.MAIN_SIM
TAGS = cf.GROUP_OURS        # ['ours'] — 우리 모델
# TAGS = cf.GROUP_ACM       #   ['acm'] — 대조군(우리 주장의 분모)
# TAGS = cf.GROUP_OURS + cf.GROUP_ACM      #   둘 다 (8잡)
# TAGS = cf.GROUP_BASELINE  #   act·diffusion·smolvla·acm2 (16잡)
# TAGS = cf.GROUP_ABLATION  #   acm_carry·acm_bimamba·acm_s7 (12잡)

SEEDS = cf.MAIN_SEEDS       # [0,1,2,3] — 4 seed 동시
GPUS  = cf.v23.available_gpus()
REPS = list(range(cf.EVAL_REPEATS))   # 5회
N_EP = cf.EVAL_N_EP                   # 500 에피소드

print('task:', TASK, '| ckpt', f'{cf.CKPT_STEP:,}', '| reps', REPS, '| n_ep', N_EP)
print('모델:', TAGS, '| seeds:', SEEDS, '| GPU:', GPUS)
print('run :', len(TAGS) * len(SEEDS) * len(REPS),
      f'(= 에피소드 {len(TAGS)*len(SEEDS)*len(REPS)*N_EP:,})')

## 사전 확인 — 150k 체크포인트. **X 가 있으면 eval 하지 말 것**
(150k 에 못 간 모델을 평가하면 학습량이 다른 것끼리 비교하게 된다)

In [ ]:
ok = cf.print_ckpt_status(TAGS, SEEDS, TASK)
print('\n=>', 'eval 진행 가능' if ok else '⚠️ 19a_train_all 먼저')

## 반복 eval

In [ ]:
cf.run_repeat_evals(TAGS, SEEDS, REPS, task=TASK, gpus=GPUS, n_episodes=N_EP)

## 결과 — SR (mean ± std, 20 run) + pooled Wilson CI

In [ ]:
rows = cf.sr_table(TAGS, SEEDS, REPS, task=TASK, n_episodes=N_EP,
                   csv_path=cf.OUTPUT_BASE / 'main_report' / f'sr_150k_{TASK}.csv')

## 다음
- 다른 그룹: `TAGS` 만 바꿔 `19a` → `19b` 다시 (권장 순서: `ours` → `acm` → baseline → ablation)
- 표·그림: `09_report_sr` · `10_report_jerk` · `18_report_horizon` · `11_efficiency`
